# 03 — File Operations

Reading/writing files, paths, JSON, CSV, YAML, and parsing log files line-by-line. Bread and butter for SRE scripts.

## Writing & reading text (always use `with`)

In [ ]:
# 'with' auto-closes the file even if an error happens
with open("app.log", "w") as f:        # w = write (overwrites!)
    f.write("2026-06-15 10:00:01 INFO  started\n")
    f.write("2026-06-15 10:00:02 ERROR disk full on /var\n")
    f.write("2026-06-15 10:00:03 WARN  high memory\n")

# append instead of overwrite
with open("app.log", "a") as f:        # a = append
    f.write("2026-06-15 10:00:04 ERROR timeout calling db\n")

# read whole file
with open("app.log") as f:             # default mode = read
    content = f.read()
print(content)

## Reading line-by-line (memory-safe for big logs)

In [ ]:
# iterate lines without loading the whole file
with open("app.log") as f:
    for line in f:
        line = line.rstrip("\n")       # strip trailing newline
        if "ERROR" in line:
            print(line)

print("---")

# read all lines into a list
with open("app.log") as f:
    lines = f.readlines()
print("total lines:", len(lines))

# count errors in one pass (generator expression)
with open("app.log") as f:
    error_count = sum(1 for line in f if "ERROR" in line)
print("errors:", error_count)

## Paths with pathlib (modern, cross-platform)

In [ ]:
from pathlib import Path

p = Path("app.log")
print(p.exists())
print(p.name)            # app.log
print(p.suffix)          # .log
print(p.stem)            # app
print(p.absolute())
print(p.stat().st_size, "bytes")

# build paths with / operator
logdir = Path("/var") / "log" / "app"
print(logdir)

# read/write directly
Path("note.txt").write_text("quick note\n")
print(Path("note.txt").read_text())

# glob: find files by pattern
for f in Path(".").glob("*.log"):
    print("found log:", f)

## JSON (configs, API responses, k8s manifests)

In [ ]:
import json

data = {"host": "web-01", "cpu": 87, "tags": ["frontend", "prod"]}

# dict -> JSON string
print(json.dumps(data))
print(json.dumps(data, indent=2))   # pretty-print

# write JSON to file
with open("server.json", "w") as f:
    json.dump(data, f, indent=2)

# read JSON from file
with open("server.json") as f:
    loaded = json.load(f)
print(loaded["cpu"])

# parse JSON string (e.g. from an API)
raw = '{"status": "ok", "latency_ms": 42}'
obj = json.loads(raw)
print(obj["latency_ms"])

## CSV (reports, exports, metrics dumps)

In [ ]:
import csv

rows = [
    {"host": "web-01", "cpu": 87},
    {"host": "db-01", "cpu": 45},
]

# write CSV with headers
with open("metrics.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["host", "cpu"])
    writer.writeheader()
    writer.writerows(rows)

# read CSV as dicts
with open("metrics.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row["host"], "->", row["cpu"])

## YAML (k8s/ansible/CI configs — needs `pip install pyyaml`)

In [ ]:
try:
    import yaml
    sample = """
name: web
replicas: 3
ports:
  - 80
  - 443
"""
    config = yaml.safe_load(sample)   # always safe_load untrusted YAML
    print(config)
    print(config["replicas"], config["ports"])
    print(yaml.safe_dump(config))     # back to YAML text
except ImportError:
    print("pyyaml not installed: pip install pyyaml")

## Mini real-world example: summarize a log file

In [ ]:
from collections import Counter

levels = Counter()
errors = []

with open("app.log") as f:
    for line in f:
        parts = line.split()
        if len(parts) >= 3:
            level = parts[2]           # 3rd field = level
            levels[level] += 1
            if level == "ERROR":
                errors.append(line.strip())

print("level counts:", dict(levels))
print("\nerror lines:")
for e in errors:
    print(" ", e)